# 03. Implementación y predicción de nuevas interacciones

## Objetivo

Implementar el modelo de clasificación de texto seleccionado en el notebook anterior, permitiendo predecir automáticamente la categoría de nuevas interacciones de clientes de una empresa de telecomunicaciones.

Se utilizará el pipeline de TF-IDF y Regresión Logística previamente entrenado, con el propósito de reutilizar el modelo sin repetir el proceso de entrenamiento.

## Modelo seleccionado

- Algoritmo: Regresión Logística.
- Representación de texto: TF-IDF.
- N-gramas: unigramas.
- Valor de regularización: C = 1.0.
- Categorías: 10.

## 2. Carga de librerías

Se importan las herramientas necesarias para cargar, guardar y utilizar el modelo de clasificación entrenado.

In [22]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import joblib
from sklearn.model_selection import train_test_split

In [23]:
ruta_modelo = Path("../models/modelo_clasificacion.pkl")

modelo = joblib.load(ruta_modelo)

print("Modelo cargado correctamente.")

Modelo cargado correctamente.


## 3. Función de predicción

Se define una función que recibe el texto de una interacción y utiliza el pipeline entrenado para identificar la categoría que mejor representa la consulta del cliente.

In [24]:
def predecir_categoria(texto):
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("El texto debe ser una cadena no vacía.")

    prediccion = modelo.predict([texto])

    return prediccion[0]

## 4. Prueba de predicciones

Se evalúa la función con ejemplos de interacciones nuevas para comprobar que el modelo pueda asignar una categoría automáticamente.

In [25]:
texto_prueba = "I want to upgrade my current mobile data plan."

categoria = predecir_categoria(texto_prueba)

print("Interacción:", texto_prueba)
print("Categoría predicha:", categoria)

Interacción: I want to upgrade my current mobile data plan.
Categoría predicha: Change Plans


In [26]:
interacciones = [
    "My internet connection keeps dropping.",
    "I want to cancel my subscription.",
    "How much is my monthly bill?",
    "I need to activate international roaming.",
    "I want to add another line to my account."
]

resultados = pd.DataFrame({
    "Interacción": interacciones,
    "Categoría predicha": [
        predecir_categoria(texto)
        for texto in interacciones
    ]
})

resultados

,Interacción,Categoría predicha
0,My internet connection keeps dropping.,Poor Connectivity
1,I want to cancel my subscription.,Others
2,How much is my monthly bill?,High Bill
3,I need to activate international roaming.,Roaming Plans
4,I want to add another line to my account.,Add New Line


Confianza de predicción

In [27]:
def predecir_con_confianza(texto):
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("El texto debe ser una cadena no vacía.")

    categoria = modelo.predict([texto])[0]
    probabilidades = modelo.predict_proba([texto])[0]

    confianza = max(probabilidades)

    return categoria, confianza

In [28]:
resultados_confianza = []

for texto in interacciones:
    categoria, confianza = predecir_con_confianza(texto)

    resultados_confianza.append({
        "Interacción": texto,
        "Categoría predicha": categoria,
        "Confianza": round(confianza * 100, 2)
    })

df_resultados = pd.DataFrame(resultados_confianza)

df_resultados

,Interacción,Categoría predicha,Confianza
0,My internet connection keeps dropping.,Poor Connectivity,19.01
1,I want to cancel my subscription.,Others,13.79
2,How much is my monthly bill?,High Bill,18.35
3,I need to activate international roaming.,Roaming Plans,33.07
4,I want to add another line to my account.,Add New Line,40.29


### Interpretación de las predicciones

Se realizaron cinco pruebas con interacciones nuevas para evaluar el comportamiento del modelo.

Cuatro de las cinco predicciones fueron coherentes con la intención de los mensajes. Se identificó una confusión en la categoría Cancel Service, que fue clasificada como Others.

También se analizaron las probabilidades asociadas a cada predicción. Los valores obtenidos fueron relativamente bajos, con un máximo de 40.29 %, lo que evidencia que el modelo presenta incertidumbre al clasificar algunas interacciones.

Estos resultados deben interpretarse teniendo en cuenta el tamaño reducido del dataset y las limitaciones del modelo. Las probabilidades obtenidas no garantizan que una predicción sea correcta.

## 5. Limitaciones

- El modelo fue entrenado con un dataset público de Kaggle de solo 103 registros, por lo que su capacidad de generalización es limitada.
- Algunas categorías presentan similitudes semánticas que pueden generar confusiones en las predicciones.
- Las probabilidades obtenidas mediante Regresión Logística no están calibradas y no deben interpretarse como garantías de precisión.
- El modelo trabaja con interacciones en inglés, de acuerdo con el idioma del dataset utilizado.
- Las predicciones requieren revisión cuando se utilizan en escenarios donde una clasificación incorrecta pueda afectar la atención al cliente.

## 6. Conclusiones

- Se implementó una función para reutilizar el pipeline de clasificación previamente entrenado.
- Se logró cargar el modelo desde un archivo serializado sin repetir el entrenamiento.
- Se realizaron pruebas con nuevas interacciones para comprobar el funcionamiento de las predicciones.
- Se incorporó el análisis de probabilidades para observar el nivel de incertidumbre de las clasificaciones.
- Las pruebas evidenciaron que el modelo puede identificar diferentes intenciones, aunque todavía presenta dificultades con categorías similares.

El proyecto constituye una base experimental para la clasificación automática de interacciones de clientes mediante técnicas de NLP y Machine Learning.